# **Imports**

In [3]:
import pandas as pd
import networkx as nx
import heapq
import math
import time

# **Cleaning and Downloading Datasets**

In [4]:
#Download and Clean Datasets
#Function that cleans edge-list dataset
def clean_dataset(file_path, separator='\t'):

    #Readsdataset csv
    df = pd.read_csv(
        file_path,
        sep=separator,
        comment='#',      #Ignores comments in Network Repository files
        header=None
    )

    #Keep only first two columns (source, target)
    df = df.iloc[:, :2]

    #Renames columns
    df.columns = ['source', 'target']

    #Removes duplicate edges
    df = df.drop_duplicates()

    #Removes self-loops (node connected to itself)
    df = df[df['source'] != df['target']]

    #Adds weight = 1 if dataset has no weights
    df['weight'] = 1

    #Drops all missing values
    df = df.dropna()

    print("Number of edges:", len(df))

    return df

movielens = clean_dataset("rec-MovieLens100K.txt")

#Saves cleaned dataset
movielens.to_csv(
    "rec-MovieLens100K_clean.csv",
    index=False
)

print(movielens.head())

#Builds graph and keeps only largest component
G = nx.from_pandas_edgelist(
    movielens,
    source='source',
    target='target',
    edge_attr='weight'
)

largest_component = max(nx.connected_components(G), key=len)

G_clean = G.subgraph(largest_component).copy()

print("Nodes:", G_clean.number_of_nodes())
print("Edges:", G_clean.number_of_edges())

#Saves final cleaned graph
nx.write_weighted_edgelist(
    G_clean,
    "MovieLens_final.edgelist"
)

#Load and clean Amazon Books dataset
amazon_books = clean_dataset("rec-AmazonBooks.txt")

#Save cleaned dataset
amazon_books.to_csv(
    "rec-AmazonBooks_clean.csv",
    index=False
)

#Builds graph
G_amazon = nx.from_pandas_edgelist(
    amazon_books,
    source='source',
    target='target',
    edge_attr='weight'
)

# Keep only largest connected component
largest_component = max(nx.connected_components(G_amazon), key=len)
G_amazon = G_amazon.subgraph(largest_component).copy()

print("Amazon Books Dataset")
print("Nodes:", G_amazon.number_of_nodes())
print("Edges:", G_amazon.number_of_edges())

nx.write_weighted_edgelist(G_amazon, "AmazonBooks_final.edgelist")

#Load and clean Netflix dataset
netflix = clean_dataset("rec-Netflix.txt")

#Save cleaned dataset
netflix.to_csv(
    "rec-Netflix_clean.csv",
    index=False
)

#Build graph
G_netflix = nx.from_pandas_edgelist(
    netflix,
    source='source',
    target='target',
    edge_attr='weight'
)

largest_component = max(nx.connected_components(G_netflix), key=len)
G_netflix = G_netflix.subgraph(largest_component).copy()

print("Netflix Dataset")
print("Nodes:", G_netflix.number_of_nodes())
print("Edges:", G_netflix.number_of_edges())

nx.write_weighted_edgelist(G_netflix, "Netflix_final.edgelist")

# Load and clean Yelp dataset
yelp = clean_dataset("rec-Yelp.txt")

# Save cleaned dataset
yelp.to_csv(
    "rec-Yelp_clean.csv",
    index=False
)

# Build graph
G_yelp = nx.from_pandas_edgelist(
    yelp,
    source='source',
    target='target',
    edge_attr='weight'
)

largest_component = max(nx.connected_components(G_yelp), key=len)
G_yelp = G_yelp.subgraph(largest_component).copy()

print("Yelp Dataset")
print("Nodes:", G_yelp.number_of_nodes())
print("Edges:", G_yelp.number_of_edges())

nx.write_weighted_edgelist(G_yelp, "Yelp_final.edgelist")

#Load and clean Amazon Full dataset
amazon_full = clean_dataset("rec-AmazonFull.txt")

#Save cleaned dataset
amazon_full.to_csv(
    "rec-AmazonFull_clean.csv",
    index=False
)

#Builds graph
G_amazon_full = nx.from_pandas_edgelist(
    amazon_full,
    source='source',
    target='target',
    edge_attr='weight'
)

largest_component = max(nx.connected_components(G_amazon_full), key=len)
G_amazon_full = G_amazon_full.subgraph(largest_component).copy()

print("Amazon Full Dataset")
print("Nodes:", G_amazon_full.number_of_nodes())
print("Edges:", G_amazon_full.number_of_edges())

nx.write_weighted_edgelist(G_amazon_full, "AmazonFull_final.edgelist")

FileNotFoundError: [Errno 2] No such file or directory: 'rec-MovieLens100K.txt'

# **Algorithm Definition**

Algorithm 1: Dijkstra Binary Heap

In [ ]:
def dijkstra_binary_heap(G, source):
    dist    = {node: math.inf for node in G.nodes()}
    prev    = {node: None     for node in G.nodes()}
    dist[source] = 0.0
    heap    = [(0.0, source)]
    visited = set()
    order   = []

    while heap:
        d, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)
        order.append(u)
        for v in G.neighbors(u):
            w = G[u][v].get("weight", 1.0)
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(heap, (dist[v], v))

    return dist, prev, order

Algorithm 2:

In [ ]:
import heapq
import math

def heuristic(a, b):
    x1, y1 = a
    x2, y2 = b

    return math.sqrt(
        (x1 - x2)**2 +
        (y1 - y2)**2
    )


def astar(G, start, goal, positions):

    open_set = []

    heapq.heappush(
        open_set,
        (0, start)
    )

    g_score = {
        node: float("inf")
        for node in G.nodes()
    }

    g_score[start] = 0

    f_score = {
        node: float("inf")
        for node in G.nodes()
    }

    f_score[start] = heuristic(
        positions[start],
        positions[goal]
    )

    prev = {}
    visited_order = []
    closed = set()

    while open_set:

        _, current = heapq.heappop(open_set)

        if current in closed:
            continue

        closed.add(current)

        visited_order.append(current)

        if current == goal:
            break

        for neighbor in G.neighbors(current):

            weight = G[current][neighbor]["weight"]

            tentative = g_score[current] + weight

            if tentative < g_score[neighbor]:

                g_score[neighbor] = tentative

                prev[neighbor] = current

                f_score[neighbor] = (
                    tentative
                    + heuristic(
                        positions[neighbor],
                        positions[goal]
                    )
                )

                heapq.heappush(
                    open_set,
                    (f_score[neighbor], neighbor)
                )

    return g_score, prev, visited_order

Algorithm 3:

Algorithm 4: Grid Graph

In [ ]:
def build_grid(G):
    nodes = list(G.nodes())
    rows  = math.ceil(len(nodes) / 2)
    node_to_cell = {n: (i % rows, i // rows) for i, n in enumerate(nodes)}
    cell_to_node = {v: k for k, v in node_to_cell.items()}
    Grid = nx.grid_2d_graph(rows, 2)
    for (r1,c1),(r2,c2) in Grid.edges():
        n1 = cell_to_node.get((r1,c1))
        n2 = cell_to_node.get((r2,c2))
        w  = G[n1][n2].get("weight", 1.0) if (n1 and n2 and G.has_edge(n1,n2)) else 1.0
        Grid[(r1,c1)][(r2,c2)]["weight"] = w
    return Grid, node_to_cell

def grid_sssp(Grid, source):
    dist    = {node: math.inf for node in Grid.nodes()}
    prev    = {node: None     for node in Grid.nodes()}
    dist[source] = 0.0
    heap    = [(0.0, source)]
    visited = set()
    order   = []

    while heap:
        d, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)
        order.append(u)
        for v in Grid.neighbors(u):
            w = Grid[u][v].get("weight", 1.0)
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                heapq.heappush(heap, (dist[v], v))

    return dist, prev, order

Algorithm 5:

In [ ]:
import networkx as nx
import heapq


class ContractionHierarchy:

    def __init__(self, G):

        self.original_graph = G
        self.graph = G.copy()

        self.rank = {}
        self.shortcuts = {}

        self.up_graph = nx.DiGraph()

    # Node Importance


    def node_importance(self, node):

        neighbors = list(self.graph.neighbors(node))

        shortcut_cover = len(neighbors)

        contracted_neighbors = 0

        for n in neighbors:
            if n in self.rank:
                contracted_neighbors += 1

        importance = shortcut_cover + contracted_neighbors

        return importance

    # Priority Queue

    def build_priority_queue(self):

        pq = []

        for node in self.graph.nodes():

            importance = self.node_importance(node)

            heapq.heappush(
                pq,
                (importance, node)
            )

        return pq


    # Witness Search


    def witness_search(
        self,
        source,
        target,
        avoid_node,
        max_cost
    ):

        pq = [(0, source)]

        distances = {source: 0}

        visited = set()

        while pq:

            dist, current = heapq.heappop(pq)

            if current in visited:
                continue

            visited.add(current)

            if current == target:
                return dist

            if dist > max_cost:
                break

            for neighbor in self.graph.neighbors(current):

                if neighbor == avoid_node:
                    continue

                weight = self.graph[current][neighbor].get(
                    "weight",
                    1
                )

                new_dist = dist + weight

                if new_dist < distances.get(
                    neighbor,
                    float("inf")
                ):

                    distances[neighbor] = new_dist

                    heapq.heappush(
                        pq,
                        (new_dist, neighbor)
                    )

        return float("inf")


    # Shortcut Generation


    def generate_shortcuts(self, node):

        shortcuts = []

        neighbors = list(self.graph.neighbors(node))

        for u in neighbors:

            for v in neighbors:

                if u == v:
                    continue

                weight_u = self.graph[u][node].get(
                    "weight",
                    1
                )

                weight_v = self.graph[node][v].get(
                    "weight",
                    1
                )

                shortcut_cost = weight_u + weight_v

                witness_cost = self.witness_search(
                    u,
                    v,
                    node,
                    shortcut_cost
                )

                if witness_cost > shortcut_cost:

                    shortcuts.append(
                        (u, v, shortcut_cost, node)
                    )

        return shortcuts

    # Contract Node


    def contract_node(self, node):

        shortcuts = self.generate_shortcuts(node)

        for u, v, cost, middle in shortcuts:

            if self.graph.has_edge(u, v):

                current = self.graph[u][v].get(
                    "weight",
                    float("inf")
                )

                if cost < current:

                    self.graph[u][v]["weight"] = cost

            else:

                self.graph.add_edge(
                    u,
                    v,
                    weight=cost
                )

            self.shortcuts[(u, v)] = (
                middle,
                cost
            )

        self.graph.remove_node(node)


    # Build Upward Graph


    def build_upward_graph(self):

        self.up_graph = nx.DiGraph()

        for u, v, data in self.original_graph.edges(data=True):

            ru = self.rank[u]
            rv = self.rank[v]

            weight = data.get(
                "weight",
                1
            )

            if ru < rv:

                self.up_graph.add_edge(
                    u,
                    v,
                    weight=weight
                )

            else:

                self.up_graph.add_edge(
                    v,
                    u,
                    weight=weight
                )

        for (u, v), (_, weight) in self.shortcuts.items():

            if u not in self.rank:
                continue

            if v not in self.rank:
                continue

            ru = self.rank[u]
            rv = self.rank[v]

            if ru < rv:

                self.up_graph.add_edge(
                    u,
                    v,
                    weight=weight
                )

            else:

                self.up_graph.add_edge(
                    v,
                    u,
                    weight=weight
                )


    # Preprocessing


    def preprocess(self):

        pq = self.build_priority_queue()

        rank_counter = 0

        while pq:

            _, node = heapq.heappop(pq)

            if node not in self.graph:
                continue

            self.rank[node] = rank_counter

            rank_counter += 1

            self.contract_node(node)

        self.build_upward_graph()

        print("CH preprocessing complete.")


    # Query


    def query(self, source, target):

        pq = [(0, source)]

        dist = {
            source: 0
        }

        prev = {}

        visited_order = []

        while pq:

            d, u = heapq.heappop(pq)

            if d > dist[u]:
                continue

            visited_order.append(u)

            if u == target:
                break

            for v in self.up_graph.neighbors(u):

                weight = self.up_graph[u][v]["weight"]

                nd = d + weight

                if nd < dist.get(
                    v,
                    float("inf")
                ):

                    dist[v] = nd

                    prev[v] = u

                    heapq.heappush(
                        pq,
                        (nd, v)
                    )

        return dist, prev, visited_order

Algorithm 6:

In [ ]:
results = []

# **REC-MOVIELENS100K**

In [ ]:
G_movielens = nx.read_weighted_edgelist("MovieLens_final.edgelist")
print(f"Nodes: {G_movielens.number_of_nodes():,}")
print(f"Edges: {G_movielens.number_of_edges():,}")

src = next(iter(G_movielens.nodes()))

dist_d, prev_d, order_d = dijkstra_binary_heap(G_movielens, src)
print("Dijkstra Binary Heap — MovieLens100K: Done")

goal = list(G_movielens.nodes())[-1]
positions = nx.spring_layout(G_movielens,seed=42)
dist_a, prev_a, order_a = astar(G_movielens,src,goal,positions)
print("A* Search — MovieLens100K: Done")

Grid_ml, node_to_cell_ml = build_grid(G_movielens)
dist_g, prev_g, order_g  = grid_sssp(Grid_ml, node_to_cell_ml[src])
print("Grid Graph SSSP   — MovieLens100K: Done")

ch_ml = ContractionHierarchy(G_movielens)
ch_ml.preprocess()
dist_ch_ml, prev_ch_ml, order_ch_ml = ch_ml.query(src,goal)
print("Contraction Hierarchies — MovieLens100K: Done")




# **REC-AMAZONBOOKS**

In [ ]:
G_amazon = nx.read_weighted_edgelist("AmazonBooks_final.edgelist")
print(f"Nodes: {G_amazon.number_of_nodes():,}")
print(f"Edges: {G_amazon.number_of_edges():,}")

src = next(iter(G_amazon.nodes()))

dist_d, prev_d, order_d = dijkstra_binary_heap(G_amazon, src)
print("Dijkstra Binary Heap — AmazonBooks: Done")

goal = list(G_amazon.nodes())[-1]
positions = nx.spring_layout(G_amazon,seed=42)
dist_a, prev_a, order_a = astar(G_amazon,src,goal,positions)
print("A* Search — AmazonBooks: Done")

Grid_ab, node_to_cell_ab = build_grid(G_amazon)
dist_g, prev_g, order_g  = grid_sssp(Grid_ab, node_to_cell_ab[src])
print("Grid Graph SSSP   — AmazonBooks: Done")

ch_ab = ContractionHierarchy(G_amazon)
ch_ab.preprocess()
dist_ch_ab, prev_ch_ab, order_ch_ab = ch_ab.query(src,goal)
print("Contraction Hierarchies — AmazonBooks: Done")

# **REC-NETFLIX**

In [ ]:
G_netflix = nx.read_weighted_edgelist("Netflix_final.edgelist")
print(f"Nodes: {G_netflix.number_of_nodes():,}")
print(f"Edges: {G_netflix.number_of_edges():,}")

src = next(iter(G_netflix.nodes()))

dist_d, prev_d, order_d = dijkstra_binary_heap(G_netflix, src)
print("Dijkstra Binary Heap — Netflix: Done")

goal = list(G_netflix.nodes())[-1]
positions = nx.spring_layout(G_netflix,seed=42)
dist_a, prev_a, order_a = astar(G_netflix,src,goal,positions)
print("A* Search — Netflix: Done")

Grid_nf, node_to_cell_nf = build_grid(G_netflix)
dist_g, prev_g, order_g  = grid_sssp(Grid_nf, node_to_cell_nf[src])
print("Grid Graph SSSP   — Netflix: Done")

ch_nf = ContractionHierarchy(G_netflix)
ch_nf.preprocess()
dist_ch_nf, prev_ch_nf, order_ch_nf = ch_nf.query(src,goal)
print("Contraction Hierarchies — Netflix: Done")

# **REC-YELP**

In [ ]:
G_yelp = nx.read_weighted_edgelist("Yelp_final.edgelist")
print(f"Nodes: {G_yelp.number_of_nodes():,}")
print(f"Edges: {G_yelp.number_of_edges():,}")

src = next(iter(G_yelp.nodes()))

dist_d, prev_d, order_d = dijkstra_binary_heap(G_yelp, src)
print("Dijkstra Binary Heap — Yelp: Done")


goal = list(G_yelp.nodes())[-1]
positions = nx.spring_layout(G_yelp,seed=42)
dist_a, prev_a, order_a = astar(G_yelp,src,goal,positions)
print("A* Search — Yelp: Done")

Grid_yl, node_to_cell_yl = build_grid(G_yelp)
dist_g, prev_g, order_g  = grid_sssp(Grid_yl, node_to_cell_yl[src])
print("Grid Graph SSSP   — Yelp: Done")

ch_yl = ContractionHierarchy(G_yelp)
ch_yl.preprocess()
dist_ch_yl, prev_ch_yl, order_ch_yl = ch_yl.query(src,goal)
print("Contraction Hierarchies — Yelp: Done")

# **REC-AMAZONFULL**


In [ ]:
G_amazonfull = nx.read_weighted_edgelist("AmazonFull_final.edgelist")
print(f"Nodes: {G_amazonfull.number_of_nodes():,}")
print(f"Edges: {G_amazonfull.number_of_edges():,}")

src = next(iter(G_amazonfull.nodes()))

dist_d, prev_d, order_d = dijkstra_binary_heap(G_amazonfull, src)
print("Dijkstra Binary Heap — AmazonFull: Done")

goal = list(G_amazonfull.nodes())[-1]
positions = nx.spring_layout(G_amazonfull,seed=42)
dist_a, prev_a, order_a = astar(G_amazonfull,src,goal,positions)
print("A* Search — AmazonFull: Done")

Grid_af, node_to_cell_af = build_grid(G_amazonfull)
dist_g, prev_g, order_g  = grid_sssp(Grid_af, node_to_cell_af[src])
print("Grid Graph SSSP   — AmazonFull: Done")

ch_af = ContractionHierarchy(G_amazonfull)
ch_af.preprocess()
dist_ch_af, prev_ch_af, order_ch_af = ch_af.query(src,goal)
print("Contraction Hierarchies — AmazonFull: Done")

# Execution Time Results

In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    results,
    columns=[
        "Dataset",
        "A* Time (s)",
        "CH Time (s)"
    ]
)

results_df

# Timing and Animation Utilities

In [ ]:
import time

def measure_algorithm(func, *args, **kwargs):
    start = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - start
    return result, elapsed


In [ ]:
from matplotlib.animation import FuncAnimation
import matplotlib.pyplot as plt
import networkx as nx

def animate_sssp(G, order, filename, max_nodes=300):
    H = G.copy()

    if H.number_of_nodes() > max_nodes:
        nodes = list(H.nodes())[:max_nodes]
        H = H.subgraph(nodes).copy()
        order = [n for n in order if n in H]

    pos = nx.spring_layout(H, seed=42)

    fig, ax = plt.subplots(figsize=(6,6))

    def update(frame):
        ax.clear()
        visited = set(order[:frame+1])

        colors = [
            "red" if n in visited else "lightgray"
            for n in H.nodes()
        ]

        nx.draw(
            H,
            pos,
            node_size=40,
            node_color=colors,
            with_labels=False,
            ax=ax
        )

        ax.set_title(f"Visited: {frame+1}")

    anim = FuncAnimation(
        fig,
        update,
        frames=len(order),
        interval=50,
        repeat=False
    )

    anim.save(filename, writer="ffmpeg")

    plt.close(fig)


# Example timing and animation usage

In [ ]:
# Example:
#
# (dist_a, prev_a, order_a), astar_time = measure_algorithm(
#     astar,
#     G_movielens,
#     src,
#     goal,
#     positions
# )
#
# animate_sssp(
#     G_movielens,
#     order_a,
#     "movielens_astar.mp4"
# )


# Final Evaluation: Timing, Results Table, and MP4 Generation

In [ ]:
results = []

datasets = [
    ("MovieLens", G_movielens),
    ("AmazonBooks", G_amazon),
    ("Netflix", G_netflix),
    ("Yelp", G_yelp),
    ("AmazonFull", G_amazonfull)
]

all_orders = {}

for name, G in datasets:

    print(f"\n========== {name} ==========")

    src = next(iter(G.nodes()))
    goal = list(G.nodes())[-1]

    positions = nx.spring_layout(
        G,
        seed=42
    )

    start = time.perf_counter()

    dist_a, prev_a, order_a = astar(
        G,
        src,
        goal,
        positions
    )

    astar_time = time.perf_counter() - start

    print(f"A*: {astar_time:.4f} s")

    start = time.perf_counter()

    ch = ContractionHierarchy(G)

    ch.preprocess()

    dist_ch, prev_ch, order_ch = ch.query(
        src,
        goal
    )

    ch_time = time.perf_counter() - start

    print(f"CH: {ch_time:.4f} s")

    results.append([
        name,
        astar_time,
        ch_time
    ])

    all_orders[name] = {
        "A*": order_a,
        "CH": order_ch
    }


In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    results,
    columns=[
        "Dataset",
        "A* Time (s)",
        "CH Time (s)"
    ]
)

results_df

In [ ]:
animate_sssp(
    G_movielens,
    all_orders["MovieLens"]["A*"],
    "movielens_astar.mp4"
)

animate_sssp(
    G_movielens,
    all_orders["MovieLens"]["CH"],
    "movielens_ch.mp4"
)

animate_sssp(
    G_amazon,
    all_orders["AmazonBooks"]["A*"],
    "amazonbooks_astar.mp4"
)

animate_sssp(
    G_amazon,
    all_orders["AmazonBooks"]["CH"],
    "amazonbooks_ch.mp4"
)

animate_sssp(
    G_netflix,
    all_orders["Netflix"]["A*"],
    "netflix_astar.mp4"
)

animate_sssp(
    G_netflix,
    all_orders["Netflix"]["CH"],
    "netflix_ch.mp4"
)

animate_sssp(
    G_yelp,
    all_orders["Yelp"]["A*"],
    "yelp_astar.mp4"
)

animate_sssp(
    G_yelp,
    all_orders["Yelp"]["CH"],
    "yelp_ch.mp4"
)

animate_sssp(
    G_amazonfull,
    all_orders["AmazonFull"]["A*"],
    "amazonfull_astar.mp4"
)

animate_sssp(
    G_amazonfull,
    all_orders["AmazonFull"]["CH"],
    "amazonfull_ch.mp4"
)

print("All MP4 files generated.")
